# Bottleneck Hub Analysis

In [32]:
import pandas as pd

df = pd.read_csv(
    r"D:\Projects\Delivery_ETA\data\delivery_data.csv"
)

print(df.shape)

(144867, 24)


In [33]:
agg_dict = {
    "route_type": "first",
    "trip_creation_time": "first",
    "od_start_time": "first",
    "od_end_time": "first",
    "actual_time": "max",
    "osrm_time": "max",
    "osrm_distance": "max",
    "actual_distance_to_destination": "max",
    "data": "first"
}

leg_df = (
    df.groupby(
        [
            "trip_uuid",
            "source_center",
            "destination_center"
        ],
        as_index=False
    )
    .agg(agg_dict)
)

print(leg_df.shape)

(26368, 12)


In [34]:
leg_df.to_csv(
    r"D:\Projects\Delivery_ETA\outputs\leg_level_data.csv",
    index=False
)

print("Saved Successfully")

Saved Successfully


In [35]:

print(leg_df.columns.tolist())

['trip_uuid', 'source_center', 'destination_center', 'route_type', 'trip_creation_time', 'od_start_time', 'od_end_time', 'actual_time', 'osrm_time', 'osrm_distance', 'actual_distance_to_destination', 'data']


In [36]:
leg_df.head()

,trip_uuid,source_center,destination_center,route_type,trip_creation_time,od_start_time,od_end_time,actual_time,osrm_time,osrm_distance,actual_distance_to_destination,data
0,trip-153671041653548748,IND209304AAA,IND000000ACB,FTL,2018-09-12 00:00:16.535741,2018-09-12 16:39:46.858469,2018-09-13 13:40:23.123744,732.0,349.0,446.5496,383.759164,training
1,trip-153671041653548748,IND462022AAA,IND209304AAA,FTL,2018-09-12 00:00:16.535741,2018-09-12 00:00:16.535741,2018-09-12 16:39:46.858469,830.0,394.0,544.8027,440.973689,training
2,trip-153671042288605164,IND561203AAB,IND562101AAA,Carting,2018-09-12 00:00:22.886430,2018-09-12 02:03:09.655591,2018-09-12 03:01:59.598855,47.0,26.0,28.1994,24.644021,training
3,trip-153671042288605164,IND572101AAA,IND561203AAB,Carting,2018-09-12 00:00:22.886430,2018-09-12 00:00:22.886430,2018-09-12 02:03:09.655591,96.0,42.0,56.9116,48.542890,training
4,trip-153671043369099517,IND000000ACB,IND160002AAC,FTL,2018-09-12 00:00:33.691250,2018-09-14 03:40:17.106733,2018-09-14 17:34:55.442454,611.0,212.0,281.2109,242.309306,training


In [37]:
import networkx as nx

train_df = leg_df[
    leg_df["data"] == "training"
].copy()

G = nx.DiGraph()

for (src, dst), group in train_df.groupby(
    ["source_center", "destination_center"]
):

    delay_ratio = (
        group["actual_time"].median()
        /
        group["osrm_time"].median()
    )

    G.add_edge(
        src,
        dst,
        weight=delay_ratio,
        trip_count=len(group)
    )

print(
    "Nodes:", G.number_of_nodes()
)

print(
    "Edges:", G.number_of_edges()
)

Nodes: 1590
Edges: 2508


## Centrality Metrics

In [38]:
degree_centrality = nx.degree_centrality(G)

print(
    "Degree Centrality Computed"
)

Degree Centrality Computed


In [39]:
betweenness_centrality = (
    nx.betweenness_centrality(
        G,
        k=200,
        normalized=True,
        seed=42
    )
)

print(
    "Betweenness Computed"
)

Betweenness Computed


In [40]:
node_metrics = pd.DataFrame({
    "hub": list(G.nodes()),
    "degree_centrality": [
        degree_centrality[n]
        for n in G.nodes()
    ],
    "betweenness_centrality": [
        betweenness_centrality[n]
        for n in G.nodes()
    ]
})

top_betweenness = (
    node_metrics
    .sort_values(
        "betweenness_centrality",
        ascending=False
    )
    .head(15)
)

top_betweenness

,hub,degree_centrality,betweenness_centrality
23,IND000000ACB,0.056010,0.234165
10,IND562132AAA,0.044682,0.099364
64,IND712311AAA,0.026432,0.092008
61,IND501359AAE,0.035872,0.067479
22,IND160002AAC,0.036501,0.057364
1323,IND781018AAB,0.018250,0.050492
58,IND421302AAG,0.033984,0.046876
56,IND382430AAB,0.017621,0.044914
97,IND110037AAM,0.025802,0.043056
53,IND302014AAA,0.010699,0.043040


## Hub Connectivity Analysis

In [41]:
node_metrics["in_degree"] = (
    node_metrics["hub"]
    .map(dict(G.in_degree()))
)

node_metrics["out_degree"] = (
    node_metrics["hub"]
    .map(dict(G.out_degree()))
)

node_metrics.head()

,hub,degree_centrality,betweenness_centrality,in_degree,out_degree
0,IND000000AAL,0.001259,0.000000,1,1
1,IND411033AAA,0.025802,0.035755,22,19
2,IND000000AAS,0.001259,0.001968,1,1
3,IND783370AAC,0.001259,0.001574,1,1
4,IND000000AAZ,0.002517,0.000781,2,2


In [42]:
node_metrics.to_csv(
    r"D:\Projects\Delivery_ETA\outputs\node_metrics.csv",
    index=False
)

print(
    node_metrics.shape
)

(1590, 5)


## Corridor Risk Assessment

In [43]:
corridor_metrics = (
    train_df
    .groupby(
        [
            "source_center",
            "destination_center"
        ]
    )
    .agg(
        trip_count=("trip_uuid","count"),
        actual_time=("actual_time","median"),
        osrm_time=("osrm_time","median")
    )
    .reset_index()
)

corridor_metrics["delay_ratio"] = (
    corridor_metrics["actual_time"]
    /
    corridor_metrics["osrm_time"]
)

corridor_metrics.head()

,source_center,destination_center,trip_count,actual_time,osrm_time,delay_ratio
0,IND000000AAL,IND411033AAA,14,68.5,26.5,2.584906
1,IND000000AAS,IND783370AAC,9,53.0,30.0,1.766667
2,IND000000AAZ,IND444203AAA,1,289.0,48.0,6.020833
3,IND000000AAZ,IND444303AAA,1,160.0,40.0,4.000000
4,IND000000ABA,IND683565AAA,5,22.0,21.0,1.047619


## High-Risk Corridors

In [44]:
top_corridors = (
    corridor_metrics[
        corridor_metrics["trip_count"] >= 5
    ]
    .sort_values(
        "delay_ratio",
        ascending=False
    )
    .head(20)
)

top_corridors

,source_center,destination_center,trip_count,actual_time,osrm_time,delay_ratio
413,IND208012AAA,IND209304AAA,8,608.0,17.0,35.764706
445,IND212402AAA,IND211002AAB,7,1047.0,36.0,29.083333
2358,IND802212AAA,IND821115AAB,5,997.0,42.0,23.738095
1091,IND425405AAA,IND424006AAA,8,1024.5,47.0,21.797872
414,IND208017AAA,IND209304AAA,5,257.0,12.0,21.416667
223,IND134109AAA,IND160002AAC,9,447.0,36.0,12.416667
2454,IND844505AAB,IND842001AAA,11,625.0,52.0,12.019231
2201,IND751002AAB,IND754103AAA,6,444.5,42.0,10.583333
2466,IND847223AAA,IND842001AAA,9,685.0,65.0,10.538462
853,IND395001AAA,IND395023AAD,9,146.0,14.0,10.428571


## Hub-Level SLA Contribution

In [45]:
train_df["sla_breach"] = (
    train_df["actual_time"]
    >
    1.2 * train_df["osrm_time"]
)

print(
    train_df["sla_breach"]
    .mean() * 100
)

94.9596242149153


In [46]:
hub_sla = (
    train_df.groupby(
        "source_center"
    )
    .agg(
        trips=("trip_uuid","count"),
        breaches=("sla_breach","sum")
    )
    .reset_index()
)

hub_sla["breach_rate"] = (
    hub_sla["breaches"]
    /
    hub_sla["trips"]
)

print(hub_sla.shape)

(1425, 4)


In [47]:
hub_sla = hub_sla.merge(
    node_metrics,
    left_on="source_center",
    right_on="hub",
    how="left"
)

top_bottlenecks = (
    hub_sla[
        hub_sla["trips"] >= 20
    ]
    .sort_values(
        [
            "betweenness_centrality",
            "breach_rate"
        ],
        ascending=False
    )
    .head(15)
)

top_bottlenecks[
    [
        "source_center",
        "trips",
        "breach_rate",
        "betweenness_centrality"
    ]
]

,source_center,trips,breach_rate,betweenness_centrality
7,IND000000ACB,777,0.974260,0.234165
871,IND562132AAA,574,0.970383,0.099364
1165,IND712311AAA,212,1.000000,0.092008
709,IND501359AAE,251,1.000000,0.067479
125,IND160002AAC,296,0.945946,0.057364
1282,IND781018AAB,72,0.902778,0.050492
568,IND421302AAG,554,1.000000,0.046876
450,IND382430AAB,203,0.960591,0.044914
26,IND110037AAM,185,0.967568,0.043056
322,IND302014AAA,61,1.000000,0.043040


In [48]:
top_bottlenecks.to_csv(
    r"D:\Projects\Delivery_ETA\outputs\top_bottlenecks.csv",
    index=False
)

corridor_metrics.to_csv(
    r"D:\Projects\Delivery_ETA\outputs\corridor_metrics.csv",
    index=False
)

print("Saved Successfully")

Saved Successfully
